# **Introduction to Lakehouse runtime Hive catalog**

This notebook showcases the Lakehouse runtime Hive catalog with a minimum viable sample.

## 1. Variable and configurations



Configure environment variables. Provide your project ID and a [region](https://cloud.google.com/bigquery/docs/locations#regions) to store your resources, such as `us-central1`.

In [ ]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
LOCATION = "us-central1"
STAGE_BUCKET_NAME = f"froyo-lakehouse-staging-{PROJECT_NBR}"
LAKEHOUSE_BUCKET_NAME = f"froyo_hive_lakehouse_catalog_{PROJECT_NBR}"
HIVE_CATALOG_NAME="froyo_hive_catalog"
APP_NAME="froyo_app"

## 2. Create a Hive warehouse storage bucket

In [ ]:
! gsutil mb -p {PROJECT_ID} -l {LOCATION} gs://{LAKEHOUSE_BUCKET_NAME}

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Creating gs://froyo_hive_lakehouse_catalog_30466744069/...
ServiceException: 409 A Cloud Storage bucket named 'froyo_hive_lakehouse_catalog_30466744069' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


## 3. Create the Hive catalog in Lakehouse runtime catalog service

In [ ]:
!gcloud alpha biglake hive catalogs create {HIVE_CATALOG_NAME} --location-uri=gs://{LAKEHOUSE_BUCKET_NAME} --primary-location={LOCATION} --description="froyo hive catalog" --project={PROJECT_ID}

Created hive_catalog [froyo_hive_catalog].


## 4. Create a Spark session with the Hive catalog configuration

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session


import os
os.environ['DATAPROC_SPARK_CONNECT_DEFAULT_DATASOURCE'] = ""

session = Session()
session.runtime_config.properties = {
 "spark.hive.metastore.blms.project.id": PROJECT_ID,
 "spark.hive.metastore.blms.catalog.default": HIVE_CATALOG_NAME,
 "spark.hive.metastore.warehouse.dir": LAKEHOUSE_BUCKET_NAME,
 "spark.hive.metastore.client.factory.class": "com.google.cloud.bigquery.metastore.client.BigLakeMetastoreClientFactory",
 "spark.sql.catalogImplementation": "hive"
}


spark = DataprocSparkSession.builder.dataprocSessionConfig(session).getOrCreate()
print("Spark session created successfully")

█████████████████████████████▊                                                  

Spark session created successfully


## 5. Create a database in the Hive catalog

In [ ]:
spark.sql("SHOW databases;").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
+---------+



In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS froyo_db;")
spark.sql(f"USE froyo_db;")

DataFrame[]

In [ ]:
spark.sql("SHOW databases;").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|froyo_db |
+---------+



In [ ]:
spark.sql("SHOW TABLES IN froyo_db").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



## 6. Read parquet & write to lakehouse bronze layer with table registration in Hive Catalog


In [ ]:
# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
customer_stage_df = spark.read.format("parquet").option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/customers")
customer_stage_df.show(2, truncate=False)

  0%|           0/1 Tasks

+-----------+------------------+------------------------------------------+---------+-----------+--------------------------+
|customer_id|customer_nm       |demographics                              |region_id|status     |consent_ts                |
+-----------+------------------+------------------------------------------+---------+-----------+--------------------------+
|1001       |Tracey Hickman    |{"age_bracket": "55+", "income": "Medium"}|REG-113  |Active     |2025-09-25 15:32:55.840448|
|1002       |Adrienne Zimmerman|{"age_bracket": "45-54", "income": "Low"} |REG-116  |Deactivated|2024-05-04 22:44:54.837283|
+-----------+------------------+------------------------------------------+---------+-----------+--------------------------+
only showing top 2 rows


In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "134217728") #128 MB

# Write to bronze layer / raw layer
# Coalescing as there are v v small files
customer_stage_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .partitionBy("region_id") \
    .option("path", f"gs://{LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customer_master") \
    .saveAsTable("froyo_db.b_customer_master")


  0%|           0/5 Tasks

In [ ]:
# Run some quick stats
spark.sql("select distinct status, count(*) customers from froyo_db.b_customer_master group by status").show(truncate=False)
spark.sql("select count(*) customers from froyo_db.b_customer_master").show(truncate=False)
spark.sql("select count(distinct *) distinct_customers from froyo_db.b_customer_master").show(truncate=False)
spark.sql("select * from froyo_db.b_customer_master limit 2").show(truncate=False)
spark.sql("select distinct region_id, count(*) customers from froyo_db.b_customer_master group by region_id").show(truncate=False)


  0%|           0/8 Tasks

+-----------+---------+
|status     |customers|
+-----------+---------+
|Active     |284320   |
|Deactivated|94393    |
|Pending    |94689    |
+-----------+---------+



  0%|           0/17 Tasks

+---------+
|customers|
+---------+
|473402   |
+---------+



  0%|           0/24 Tasks

+------------------+
|distinct_customers|
+------------------+
|473402            |
+------------------+



  0%|           0/1 Tasks

+-----------+----------------+----------------------------------------------+-----------+--------------------------+---------+
|customer_id|customer_nm     |demographics                                  |status     |consent_ts                |region_id|
+-----------+----------------+----------------------------------------------+-----------+--------------------------+---------+
|1120       |Jessica Robinson|{"age_bracket": "55+", "income": "Ultra-High"}|Active     |2025-01-27 12:12:35.417056|REG-110  |
|1218       |Cameron Rose    |{"age_bracket": "45-54", "income": "Medium"}  |Deactivated|2024-07-25 13:37:09.238762|REG-110  |
+-----------+----------------+----------------------------------------------+-----------+--------------------------+---------+



  0%|           0/17 Tasks

+---------+---------+
|region_id|customers|
+---------+---------+
|REG-120  |15823    |
|REG-116  |15849    |
|REG-121  |15823    |
|REG-102  |15843    |
|REG-125  |15800    |
|REG-127  |15766    |
|REG-114  |15785    |
|REG-122  |15791    |
|REG-128  |15758    |
|REG-104  |15712    |
|REG-107  |15724    |
|REG-105  |15732    |
|REG-119  |15448    |
|REG-108  |15634    |
|REG-110  |16068    |
|REG-124  |15861    |
|REG-100  |15925    |
|REG-112  |15966    |
|REG-103  |15804    |
|REG-109  |15814    |
+---------+---------+
only showing top 20 rows


## 7. Review details of database objects

In [ ]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|spark_catalog|
+-------------+



In [ ]:
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|froyo_db |
+---------+



In [ ]:
spark.sql("DESCRIBE DATABASE froyo_db").show(truncate=False)

+--------------+--------------------------------------------------------------------------------------+
|info_name     |info_value                                                                            |
+--------------+--------------------------------------------------------------------------------------+
|Catalog Name  |spark_catalog                                                                         |
|Namespace Name|froyo_db                                                                              |
|Comment       |                                                                                      |
|Location      |file:/var/dataproc/tmp/spark/work/froyo_hive_lakehouse_catalog_30466744069/froyo_db.db|
|Owner         |spark                                                                                 |
+--------------+--------------------------------------------------------------------------------------+



In [ ]:
spark.sql("SHOW TABLES IN froyo_db").show(truncate=False)

+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|froyo_db |b_customer_master|false      |
+---------+-----------------+-----------+



In [ ]:
spark.sql("DESCRIBE FORMATTED froyo_db.b_customer_master").show(40,truncate=False)

+----------------------------+------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                     |comment|
+----------------------------+------------------------------------------------------------------------------+-------+
|customer_id                 |bigint                                                                        |NULL   |
|customer_nm                 |string                                                                        |NULL   |
|demographics                |string                                                                        |NULL   |
|status                      |string                                                                        |NULL   |
|consent_ts                  |timestamp_ntz                                                                 |NULL   |
|region_id                   |string                    

## 8. Review the hive warehouse layout in Google Cloud Storage

In [ ]:
import pandas as pd
from google.cloud import storage

# Initialize a GCS client
storage_client = storage.Client(project=PROJECT_ID)

# Get the bucket
bucket = storage_client.get_bucket(LAKEHOUSE_BUCKET_NAME)

# List all blobs in the bucket
blobs = bucket.list_blobs()

# Create a list to store blob information
blob_data = []
for blob in blobs:
    blob_data.append({
        'Name': blob.name,
        'Size (bytes)': blob.size,
        'Content Type': blob.content_type,
        'Creation Time': blob.time_created,
        'Updated Time': blob.updated
    })

# Create a Pandas DataFrame from the blob data
df_gcs_objects = pd.DataFrame(blob_data)

# Display the DataFrame
print(df_gcs_objects.to_markdown(index=False))

| Name                                                                                                                   |   Size (bytes) | Content Type             | Creation Time                    | Updated Time                     |
|:-----------------------------------------------------------------------------------------------------------------------|---------------:|:-------------------------|:---------------------------------|:---------------------------------|
| froyo-raw/bronze/customer_master/                                                                                      |              0 | application/octet-stream | 2026-05-13 20:44:21.661000+00:00 | 2026-05-13 20:44:21.661000+00:00 |
| froyo-raw/bronze/customer_master/_SUCCESS                                                                              |              0 | application/octet-stream | 2026-05-13 20:44:21.861000+00:00 | 2026-05-13 20:44:21.861000+00:00 |
| froyo-raw/bronze/customer_master/region_id=REG-100

## 9. Query the table from BigQuery

You have to use the notation `PROJECT_ID.CATALOG_NAME.DATABASE_NAME.TABLE_NAME`

In [ ]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

--SELECT * FROM `lakehouse-solutions-build.spark_catalog.froyo_db.b_customer_master` LIMIT 3

SELECT * FROM `lakehouse-solutions-build.froyo_hive_catalog.froyo_db.b_customer_master` LIMIT 3

# This concludes the tutorial. Proceed back to the lab manual.